# Phase 1 — Extract UNSAFE Embeddings (Days 1–5)

Loads ToxicChat, runs **Llama-Guard-3-8B** (int8, ~9 GB on a free T4),
and saves terminal hidden states (dim=4096) for every prompt flagged **UNSAFE**.

> **GPU required.** Runtime → Change runtime type → T4 GPU.
> **HF token required.** Paste a token with access to `meta-llama/Llama-Guard-3-8B`.

### Step 0 — get the repo onto this runtime

Run once per session; pulls latest if the repo already exists.

In [ ]:
import os, subprocess
from pathlib import Path

# ── EDIT THIS if you have a GitHub remote ────────────────────────────────────
GITHUB_URL = "https://github.com/yogijoshi86/SLMProject.git"
# ─────────────────────────────────────────────────────────────────────────────

TARGET = Path("/content/SLMProject")

if TARGET.is_dir() and (TARGET / "src" / "guardrail_audit").is_dir():
    print("Repo already present — pulling latest…")
    subprocess.run(["git", "-C", str(TARGET), "pull", "--ff-only"], check=True)

elif GITHUB_URL:
    print("Cloning from GitHub…")
    subprocess.run(["git", "clone", GITHUB_URL, str(TARGET)], check=True)
    print("Cloned to", TARGET)

else:
    # ── Google Drive fallback ─────────────────────────────────────────────────
    # Mount Drive once then point DRIVE_PATH at wherever you stored the folder.
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_PATH = "/content/drive/MyDrive/SLMProject"   # adjust if needed
    if not Path(DRIVE_PATH).is_dir():
        raise FileNotFoundError(
            f"Could not find the repo at {DRIVE_PATH}. "
            "Either set GITHUB_URL above, or copy the SLMProject folder to your Drive "
            "and update DRIVE_PATH."
        )
    import shutil
    shutil.copytree(DRIVE_PATH, str(TARGET))
    print("Copied from Drive to", TARGET)

In [ ]:
import os, sys
from pathlib import Path

REPO_ROOT = Path("/content/SLMProject")
assert (REPO_ROOT / "src" / "guardrail_audit").is_dir(), \
    f"src/guardrail_audit not found under {REPO_ROOT}. Did the previous cell succeed?"

sys.path.insert(0, str(REPO_ROOT / "src"))
os.chdir(REPO_ROOT)
print("Repo root:", REPO_ROOT)

In [ ]:
# Pin exact versions proven compatible on Colab T4.
%pip install -q -e ".[quant,explainer,dev]" \
    "torch>=2.4.0" "torchvision>=0.19.0" \
    "transformers==4.44.2" \
    "accelerate==0.33.0" \
    "bitsandbytes>=0.45.0" \
    "numpy>=1.26,<2.0"

In [ ]:
# MUST RUN after INSTALL. Restarts the kernel so upgraded packages load fresh.
# After restart: skip this cell and the INSTALL cell, run from the next cell down.
import os, sys
# Sanity-check: if numpy is already broken, restart is definitely needed.
try:
    import numpy as np; np.random.seed(0)
    print("Packages loaded OK. Restarting to ensure clean state...")
except Exception as e:
    print(f"Detected stale package (numpy ABI mismatch or similar): {e}")
    print("Restarting now...")
os.kill(os.getpid(), 9)

### ↑ After that cell restarts the kernel, start from the LOCATE cell below ↓

In [ ]:
import os, sys
from pathlib import Path

REPO_ROOT = Path("/content/SLMProject")
assert (REPO_ROOT / "src" / "guardrail_audit").is_dir(), \
    f"src/guardrail_audit not found under {REPO_ROOT}. Did the previous cell succeed?"

sys.path.insert(0, str(REPO_ROOT / "src"))
os.chdir(REPO_ROOT)
print("Repo root:", REPO_ROOT)

In [ ]:
# Sanity check — if this errors, re-run the INSTALL + RESTART cells above.
import numpy as np; np.random.seed(0)
import torch; assert torch.cuda.is_available()
print("numpy", np.__version__, "| torch", torch.__version__, "| CUDA OK")

### (Optional) Cache model to Google Drive — avoids re-downloading 16 GB each session

In [ ]:
# Mount Google Drive and redirect HuggingFace cache there.
# The 16 GB model downloads once to Drive; future sessions load it in ~2 min instead of re-downloading.
# Skip this cell if you don't want Drive persistence (model re-downloads every session).
from google.colab import drive
from pathlib import Path
import os

drive.mount("/content/drive")

# Adjust this path if you want the cache in a different Drive folder.
HF_CACHE = "/content/drive/MyDrive/hf_cache"
Path(HF_CACHE).mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = HF_CACHE
os.environ["TRANSFORMERS_CACHE"] = HF_CACHE
print(f"HF cache → {HF_CACHE}")

In [ ]:
import torch
assert torch.cuda.is_available(), (
    "No CUDA GPU. In Colab: Runtime -> Change runtime type -> T4 GPU, then rerun."
)
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

### Hugging Face auth

In [ ]:
import os, getpass
from huggingface_hub import login

# Token from https://huggingface.co/settings/tokens (read access).
# Accept the license at https://huggingface.co/meta-llama/Llama-Guard-3-8B first.
token = getpass.getpass("HuggingFace token: ")
os.environ["HF_TOKEN"] = token
login(token=token)   # registers the token with transformers globally
print("Logged in.")

In [ ]:
from guardrail_audit.utils import load_config, set_seed

# colab_smoke.yaml = 500 prompts + int8 (fits a free T4). Swap to default.yaml for full runs.
CONFIG = "config/colab_smoke.yaml"
cfg = load_config(CONFIG)
set_seed(cfg.seed)
cfg

### Load prompts

In [ ]:
from guardrail_audit.data import load_prompts

records = load_prompts(
    dataset_name=cfg.data.dataset_name,
    dataset_config=cfg.data.dataset_config,
    split=cfg.data.split,
    text_column=cfg.data.text_column,
    max_samples=cfg.data.max_samples,
)
print(f"Loaded {len(records)} prompts (max_samples={cfg.data.max_samples})")
records[0]

### Load Llama-Guard-3 (downloads ~16 GB on first run)

In [ ]:
import torch
free, total = torch.cuda.mem_get_info()
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Free: {free/1e9:.1f} GB / Total: {total/1e9:.1f} GB")
if free < 9e9:
    print("WARNING: < 9 GB free. Do Runtime → Disconnect and delete runtime for a clean slate.")
else:
    print("OK — enough free RAM for int8 load.")

In [ ]:
from guardrail_audit.models import load_guard

guard = load_guard(cfg.model)
print("Loaded:", cfg.model.name, "| dtype:", cfg.model.dtype)

### Run extraction → save embeddings

In [ ]:
from guardrail_audit.extraction import extract_unsafe_embeddings

payload = extract_unsafe_embeddings(
    guard=guard,
    records=records,
    batch_size=cfg.extraction.batch_size,
    output_path=cfg.paths.embeddings,
)
print(payload["stats"])

**Next:** open `02_clustering.ipynb`.
If you hit CUDA OOM, lower `extraction.batch_size` in `config/colab_smoke.yaml` (try 2 or 1).